In [ ]:
## 1. Load a Complex Tabular Dataset
We load a complex tabular dataset containing both numerical and categorical features (such as Housing Prices or Credit Default Risk) to serve as our base data[span_0](start_span)[span_0](end_span)[span_1](start_span)[span_1](end_span).

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# Load dataset and add a categorical feature for mixed tabular data
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df['OceanProximity'] = np.random.choice(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN'], size=len(df))

print("Dataset loaded successfully. Shape:", df.shape)
df.head()

Dataset loaded successfully. Shape: (20640, 10)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,OceanProximity
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,NEAR BAY
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,INLAND
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,INLAND
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,<1H OCEAN
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,INLAND


In [ ]:
## 2. Perform Train/Test Splitting First
To strictly prevent data leakage, we split our data into training and testing sets before performing any imputation, scaling, or encoding[span_2](start_span)[span_2](end_span)[span_3](start_span)[span_3](end_span).

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# 1. Load a clean built-in dataset
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

# Add a categorical column to satisfy mixed tabular feature requirements
df['Category'] = np.random.choice(['TypeA', 'TypeB', 'TypeC'], size=len(df))

# 2. Define target column ('target' is built into this dataset)
target_column = 'target'
X = df.drop(columns=[target_column])
y = df[target_column]

# 3. Perform train/test split FIRST before any transformations
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Target column: {target_column}")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Target column: target
Training set shape: (353, 11)
Testing set shape: (89, 11)


In [ ]:
## 3. Implement sklearn ColumnTransformer & Pipeline
We build reusable Scikit-Learn pipelines using `ColumnTransformer` for numerical scaling and categorical encoding[span_6](start_span)[span_6](end_span)[span_7](start_span)[span_7](end_span).

In [3]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Numerical and categorical transformation pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# End-to-end model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))
])

model_pipeline.fit(X_train, y_train)
print(f"Test R^2 Score: {model_pipeline.score(X_test, y_test):.4f}")


Test R^2 Score: 0.4391


In [ ]:
## 4. Conduct Feature Importance Analysis
We analyze and compute feature rankings using tree-based feature importances from our trained model[span_10](start_span)[span_10](end_span)[span_11](start_span)[span_11](end_span).

In [4]:
cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + encoded_cat_features

# Extract tree-based feature importances[span_12](start_span)[span_12](end_span)[span_13](start_span)[span_13](end_span)
importances = model_pipeline.named_steps['regressor'].feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': all_features, 
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top Feature Importances:")
print(feature_importance_df.head(10))

Top Feature Importances:
  Feature  Importance
2     bmi    0.377496
8      s5    0.198325
3      bp    0.088246
9      s6    0.072935
0     age    0.053705
5      s2    0.052846
6      s3    0.048551
4      s1    0.047632
7      s4    0.028943
1     sex    0.010320
